## Introduction to GPU Acceleration
### 🔍 Why Use GPUs?

GPUs are optimized for large-scale parallel computation, making them ideal for matrix-heavy tasks in deep learning. In this lab, you'll compare training times and performance on CPU vs GPU and learn how to write GPU-efficient code.


## Checking Device Availability

## Tensorflow

In [3]:
import tensorflow as tf
print('Is GPU Available?',tf.config.list_physical_devices('GPU'))

Is GPU Available? [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


## Pytorch

In [4]:
import torch
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using Device : ", device)

Using Device :  cuda


## 🚀 Moving Models and Data to GPU

To fully utilize the GPU, both the model and input data must be moved to the GPU device. This ensures the computation is performed on the GPU instead of the CPU.

Let's see how to do this in both TensorFlow and PyTorch.

In [5]:
# TensorFlow uses GPU by default when available
import tensorflow as tf

with tf.device('/GPU:0'):  # or '/CPU:0' for CPU
    a = tf.random.normal([1000, 1000])
    b = tf.random.normal([1000, 1000])
    c = tf.matmul(a, b)
    print("Operation completed on:", c.device)

Operation completed on: /job:localhost/replica:0/task:0/device:GPU:0


## PyTorch - Manual GPU Transfer

In [6]:
import torch

# Use 'cuda' if GPU is available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Example tensor operation on GPU
a = torch.randn(1000, 1000).to(device)
b = torch.randn(1000, 1000).to(device)
c = torch.matmul(a, b)

print("Tensor 'c' is on device:", c.device)

Tensor 'c' is on device: cuda:0


##Measuring Training Time on CPU vs GPU

## ⏱️ Performance Benchmark: CPU vs GPU

We'll train a simple model on the MNIST dataset using both CPU and GPU. This will help us visualize the speedup provided by GPU acceleration.

Steps:
- Train on CPU and measure the time
- Train on GPU and measure the time
- Compare the difference


In [7]:
import time
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

# Data
transform = transforms.ToTensor()
train_data = datasets.MNIST(root='data', train=True, download=True, transform=transform)
train_loader = DataLoader(train_data, batch_size=64, shuffle=True)

# Model
class SimpleModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(784, 256)
        self.fc2 = nn.Linear(256, 10)

    def forward(self, x):
        x = x.view(-1, 784)
        x = F.relu(self.fc1(x))
        return F.log_softmax(self.fc2(x), dim=1)

100%|██████████| 9.91M/9.91M [00:00<00:00, 19.8MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 483kB/s]
100%|██████████| 1.65M/1.65M [00:00<00:00, 4.41MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 12.3MB/s]


## Train on CPU

In [9]:
# Train on CPU
def train_on_cpu():
    device = torch.device("cpu")
    model = SimpleModel().to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
    criterion = nn.NLLLoss()

    start_time = time.time()
    for epoch in range(1):  # Short training
        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)
            optimizer.zero_grad()
            output = model(images)
            loss = criterion(output, labels)
            loss.backward()
            optimizer.step()
    end_time = time.time()
    print(f"✅ CPU training time: {end_time - start_time:.2f} sec")

train_on_cpu()

✅ CPU training time: 11.53 sec


## Train on GPU
## Change your runtine to T4 GPU and run the following code block

In [8]:
import torch
# Train on GPU
def train_on_gpu():
    if not torch.cuda.is_available():
        print("🚫 CUDA not available on this system.")
        return

    device = torch.device("cuda")
    model = SimpleModel().to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
    criterion = nn.NLLLoss()

    start_time = time.time()
    for epoch in range(1):  # Short training
        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)
            optimizer.zero_grad()
            output = model(images)
            loss = criterion(output, labels)
            loss.backward()
            optimizer.step()
    end_time = time.time()
    print(f"✅ GPU training time: {end_time - start_time:.2f} sec")

train_on_gpu()

✅ GPU training time: 8.00 sec


## EXERCISE

In [1]:
import tensorflow as tf
import time
print('Is GPU Available?', tf.config.list_physical_devices('GPU'))

Is GPU Available? [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


## Quick GPU Operation Test

In [2]:
with tf.device('/GPU:0'):  # or '/CPU:0' for CPU
    a = tf.random.normal([1000, 1000])
    b = tf.random.normal([1000, 1000])
    c = tf.matmul(a, b)
    print("Operation completed on:", c.device)

Operation completed on: /job:localhost/replica:0/task:0/device:GPU:0


## Load and prepare MNIST Data

In [3]:
(x_train, y_train), (x_test, y_test) = tf.keras.datasets.mnist.load_data()

x_train = x_train.astype("float32") / 255.0
x_train = x_train.reshape(-1, 784)   # flatten images, like x.view(-1, 784) in PyTorch

train_dataset = tf.data.Dataset.from_tensor_slices((x_train, y_train))
train_dataset = train_dataset.shuffle(10000).batch(64)

11490434/11490434 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


## Defining the Model (Customer Layers)

In [4]:
# Custom hyperparameters — experiment with these
HIDDEN_UNITS_1 = 256
HIDDEN_UNITS_2 = 128   # extra layer, feel free to add/remove

def build_model():
    model = tf.keras.Sequential([
        tf.keras.layers.Input(shape=(784,)),
        tf.keras.layers.Dense(HIDDEN_UNITS_1, activation='relu'),
        tf.keras.layers.Dense(HIDDEN_UNITS_2, activation='relu'),
        tf.keras.layers.Dense(10)   # logits, softmax applied in loss
    ])
    return model

## Train on CPU

In [5]:
import time

def train_on_cpu():
    device = '/CPU:0'
    with tf.device(device):
        model = build_model()
        optimizer = tf.keras.optimizers.Adam(learning_rate=0.001)
        loss_fn = tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True)

        start_time = time.time()
        for epoch in range(3):  # Short training
            for images, labels in train_dataset:
                with tf.GradientTape() as tape:
                    output = model(images, training=True)
                    loss = loss_fn(labels, output)
                grads = tape.gradient(loss, model.trainable_variables)
                optimizer.apply_gradients(zip(grads, model.trainable_variables))
        end_time = time.time()
        print(f"✅ CPU training time: {end_time - start_time:.2f} sec")

train_on_cpu()

✅ CPU training time: 140.11 sec


## Train on GPU

In [5]:
import time

def train_on_gpu():
    gpus = tf.config.list_physical_devices('GPU')
    if not gpus:
        print("🚫 GPU not available on this system.")
        return

    device = '/GPU:0'
    with tf.device(device):
        model = build_model()
        optimizer = tf.keras.optimizers.Adam(learning_rate=0.001)
        loss_fn = tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True)

        start_time = time.time()
        for epoch in range(1):  # Short training
            for images, labels in train_dataset:
                with tf.GradientTape() as tape:
                    output = model(images, training=True)
                    loss = loss_fn(labels, output)
                grads = tape.gradient(loss, model.trainable_variables)
                optimizer.apply_gradients(zip(grads, model.trainable_variables))
        end_time = time.time()
        print(f"✅ GPU training time: {end_time - start_time:.2f} sec")

train_on_gpu()

✅ GPU training time: 36.13 sec
